# 05b – Conformal prediction robuste : calibration croisée sur les moteurs (CV+)

**Le problème :** à l'étape 5, le quantile q (26,9 cycles) a été estimé sur **15 moteurs seulement**. Avec 15 autres moteurs, il aurait pu être très différent. Les 70 moteurs d'entraînement, eux, ne servaient pas à la calibration.

**La solution :** on répartit les 85 moteurs (train + calib) en 5 groupes. Pour chaque groupe, on entraîne un LightGBM sur les 4 autres groupes et on prédit les moteurs du groupe, qu'il n'a jamais vus. On obtient ainsi une erreur « hors échantillon » pour les **85 moteurs**, qui servent alors tous à la fois à entraîner et à calibrer.

La validation (15 moteurs) et le test (100 moteurs) restent intouchés pour vérifier le résultat.

In [ ]:
import sys
from pathlib import Path
sys.path.append(str(Path.cwd().parent))

import joblib
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns

from src.data.load import PROJECT_ROOT
from src.features.preprocess import FEATURES_FD001
from src.features.rolling import add_rolling_features, last_cycle
from src.conformal.conformal import sample_cuts, conformal_quantile, coverage, mean_width
from src.conformal.cross import assign_folds, fit_fold_models, oof_predictions, fold_predictions, CVPlus, MondrianCV

sns.set_theme(style="whitegrid")
PROC = PROJECT_ROOT / "data" / "processed"
MODELS = PROJECT_ROOT / "models"
ALPHA = 0.10
N_FOLDS = 5
BINS = [-1, 25, 50, 75, 100, 125]
LABELS = ["0-25", "25-50", "50-75", "75-100", "100-125"]

## 1. Données et réglages du modèle

On reprend exactement les hyperparamètres de l'étape 3, avec le nombre d'arbres trouvé par l'arrêt anticipé.

In [ ]:
sets = {k: add_rolling_features(pd.read_csv(PROC / f"FD001_{k}.csv"), FEATURES_FD001, windows=(5, 10, 20))
        for k in ["train", "calib", "val", "test"]}
bundle = joblib.load(MODELS / "lgbm_fd001.joblib")
FEATS = bundle["features"]
PARAMS = bundle["model"].get_params()
PARAMS["n_estimators"] = bundle["model"].best_iteration_
print("Nombre d'arbres :", PARAMS["n_estimators"])

pool = pd.concat([sets["train"], sets["calib"]], ignore_index=True)
print(f"Moteurs disponibles pour entraîner ET calibrer : {pool.unit.nunique()}")

## 2. Cinq modèles, chacun sans un groupe de moteurs

Entraînement de 5 LightGBM : compter 1 à 3 minutes.

In [ ]:
folds = assign_folds(pool.unit.unique(), n_folds=N_FOLDS, seed=0)
models = fit_fold_models(pool, FEATS, folds, PARAMS)

## 3. Prédictions hors échantillon

Comme à l'étape 5, chaque moteur est coupé à 20 cycles aléatoires (au moins 31 cycles observés) pour ressembler au test. Chaque coupure est prédite par le modèle qui n'a **jamais vu ce moteur**.

In [ ]:
cal = oof_predictions(pool, FEATS, folds, models, n_cuts=20, seed=0)
cal["R"] = (cal.RUL - cal.pred_oof).abs()
print(f"{len(cal)} points de calibration ({cal.unit.nunique()} moteurs) au lieu de 300 (15 moteurs)")
print("\nRMSE hors échantillon par groupe :")
print(cal.groupby("fold").apply(lambda d: np.sqrt(((d.RUL - d.pred_oof) ** 2).mean()), include_groups=False).round(2))

## 4. Le problème illustré : q dépend-il du choix des 15 moteurs ?

Grâce aux erreurs hors échantillon des 85 moteurs, on peut simuler ce qui se serait passé à l'étape 5 avec **d'autres** tirages de 15 moteurs de calibration : on tire 1 000 fois 15 moteurs au hasard et on calcule q à chaque fois.

In [ ]:
rng = np.random.default_rng(0)
units = cal.unit.unique()
q15 = [conformal_quantile(cal[cal.unit.isin(rng.choice(units, 15, replace=False))].R, ALPHA) for _ in range(1000)]
q85 = conformal_quantile(cal.R, ALPHA)
lo5, hi95 = np.percentile(q15, [5, 95])
print(f"q avec 15 moteurs : entre {lo5:.1f} et {hi95:.1f} cycles dans 90 % des tirages")
print(f"q avec les 85 moteurs : {q85:.1f} cycles")

fig, ax = plt.subplots(figsize=(8, 4))
ax.hist(q15, bins=40, color="steelblue", alpha=0.8, label="q estimé sur 15 moteurs (1 000 tirages)")
ax.axvline(q85, color="red", lw=2, label=f"q estimé sur 85 moteurs = {q85:.1f}")
ax.axvline(26.85, color="black", ls="--", label="q de l'étape 5 = 26,9")
ax.set(xlabel="Demi-largeur q de l'intervalle à 90 % (cycles)", ylabel="Nombre de tirages", title="Variabilité de q selon les moteurs de calibration")
ax.legend()
plt.show()

## 5. Intervalles CV+

Pour une nouvelle donnée x, chacun des 1 700 points de calibration i propose un intervalle [μ₋ₖ₍ᵢ₎(x) − Rᵢ, μ₋ₖ₍ᵢ₎(x) + Rᵢ], où μ₋ₖ₍ᵢ₎ est le modèle qui n'a pas vu le moteur de i. La borne basse est le quantile α de toutes les bornes basses proposées, la borne haute le quantile 1−α des bornes hautes.

In [ ]:
val_cuts = sample_cuts(sets["val"], n_cuts=20, seed=1)
test_last = last_cycle(sets["test"])
FP = {"val": fold_predictions(models, val_cuts[FEATS]), "test": fold_predictions(models, test_last[FEATS])}
EVAL = {"val": val_cuts, "test": test_last}
for k, d in EVAL.items():
    d["pred"] = FP[k].mean(axis=1)   # prédiction ponctuelle = moyenne des 5 modèles

rows = []
for level in [0.5, 0.6, 0.7, 0.8, 0.9, 0.95]:
    cvp = CVPlus(alpha=1 - level).fit(cal.pred_oof, cal.RUL, cal.fold)
    for k, d in EVAL.items():
        lo, hi = cvp.predict(FP[k])
        rows.append({"niveau visé": level, "ensemble": k, "couverture": coverage(d.RUL, lo, hi), "largeur": mean_width(lo, hi)})
curve = pd.DataFrame(rows)
print(curve.pivot(index="niveau visé", columns="ensemble", values=["couverture", "largeur"]).round(3))

fig, ax = plt.subplots(figsize=(6, 6))
for k, d in curve.groupby("ensemble"):
    ax.plot(d["niveau visé"], d["couverture"], "o-", label=k)
ax.plot([0.45, 1], [0.45, 1], "k--", label="couverture parfaite")
ax.set(xlabel="Niveau de confiance visé", ylabel="Couverture observée", title="Couverture empirique du CV+")
ax.legend()
plt.show()

## 6. Mondrian avec les 85 moteurs

Même idée qu'à l'étape 5 (un quantile par tranche de RUL prédite), mais chaque tranche dispose maintenant d'environ 5 fois plus de points. On calcule aussi une **borne basse adaptative** par tranche.

In [ ]:
mcv = MondrianCV(alpha=ALPHA, alpha_lower=0.05).fit(cal.pred_oof, cal.RUL)
mcv.table()

## 7. Comparaison avec l'étape 5

In [ ]:
def by_bin(d, lo, hi):
    t = pd.DataFrame({"RUL": d.RUL.values, "lo": lo, "hi": hi})
    t["tranche"] = pd.cut(t.RUL, bins=BINS, labels=LABELS)
    t["couvert"] = (t.RUL >= t.lo) & (t.RUL <= t.hi)
    t["largeur"] = t.hi - t.lo
    return t.groupby("tranche", observed=True).agg(couverture=("couvert", "mean"), largeur=("largeur", "mean"), n=("RUL", "size")).reset_index()

cvp = CVPlus(alpha=ALPHA).fit(cal.pred_oof, cal.RUL, cal.fold)
new = {"CV+ (85 moteurs)": {k: cvp.predict(FP[k]) for k in EVAL},
       "Mondrian CV (85 moteurs)": {k: mcv.predict(d.pred) for k, d in EVAL.items()}}

rows, per_bin = [], []
for m, r in new.items():
    for k, d in EVAL.items():
        lo, hi = r[k]
        bb = by_bin(d, lo, hi)
        rows.append({"méthode": m, "ensemble": k, "couverture": coverage(d.RUL, lo, hi), "largeur moyenne": mean_width(lo, hi), "pire tranche": bb.couverture.min()})
        if k == "val":
            per_bin.append(bb.assign(méthode=m))
summary = pd.DataFrame(rows)

old_path = PROJECT_ROOT / "reports" / "conformal_summary.csv"
if old_path.exists():
    old = pd.read_csv(old_path)
    old["méthode"] = old["méthode"] + " (15 moteurs)"
    summary = pd.concat([old, summary], ignore_index=True)
print("Objectif : couverture ≥ 0.90 avec la largeur la plus petite possible")
summary.pivot(index="méthode", columns="ensemble", values=["couverture", "largeur moyenne", "pire tranche"]).round(3)

In [ ]:
per_bin = pd.concat(per_bin)
fig, axes = plt.subplots(1, 2, figsize=(14, 4))
sns.barplot(data=per_bin, x="tranche", y="couverture", hue="méthode", ax=axes[0])
axes[0].axhline(1 - ALPHA, color="red", ls="--")
axes[0].set(ylim=(0.5, 1.02), title="Couverture par tranche de vraie RUL (validation)", xlabel="Vraie RUL")
sns.barplot(data=per_bin, x="tranche", y="largeur", hue="méthode", ax=axes[1])
axes[1].set(title="Largeur moyenne", xlabel="Vraie RUL", ylabel="Largeur (cycles)")
plt.tight_layout()
plt.show()

## 8. Borne basse garantie : globale contre adaptative

- **Globale (étape 5)** : on recule de 26,9 cycles quelle que soit la prédiction.
- **Adaptative (Mondrian CV)** : on recule peu quand la panne est proche et que le modèle est précis, davantage en milieu de vie.

In [ ]:
lb_global = conformal_quantile(cal.pred_oof - cal.RUL, 0.05)
for k, d in EVAL.items():
    Lg = np.clip(d.pred - lb_global, 0, None)
    La = mcv.lower_bound(d.pred)
    print(f"{k:<5} | globale : respectée {np.mean(d.RUL >= Lg):.1%}, recul moyen {np.mean(d.pred - Lg):.1f} cycles"
          f"  |  adaptative : respectée {np.mean(d.RUL >= La):.1%}, recul moyen {np.mean(d.pred - La):.1f} cycles")

t = test_last.assign(L=mcv.lower_bound(test_last.pred)).sort_values("RUL").reset_index(drop=True)
fig, ax = plt.subplots(figsize=(12, 4))
ax.vlines(t.index, t.L, t.pred, color="seagreen", alpha=0.5, label="de la borne basse à la prédiction")
ax.scatter(t.index, t.RUL, s=12, color="black", zorder=3, label="vraie RUL")
ax.scatter(t.index[t.RUL < t.L], t.RUL[t.RUL < t.L], s=40, color="red", zorder=4, label="borne violée")
ax.set(xlabel="Moteurs de test (triés par vraie RUL)", ylabel="RUL (cycles)", title="Borne basse adaptative à 95 % (Mondrian CV) – jeu de test")
ax.legend()
plt.show()

## 9. Robustesse : et avec une autre répartition des moteurs ?

C'est la question à laquelle cette étape devait répondre. On refait **toute** la procédure (5 nouveaux modèles, nouvelles coupures) avec 3 répartitions différentes des moteurs en groupes. Compter 3 à 8 minutes.

In [ ]:
RUN_REPEATS = True

if RUN_REPEATS:
    rep = []
    for seed in [1, 2, 3]:
        f_s = assign_folds(pool.unit.unique(), n_folds=N_FOLDS, seed=seed)
        m_s = fit_fold_models(pool, FEATS, f_s, PARAMS, verbose=False)
        c_s = oof_predictions(pool, FEATS, f_s, m_s, n_cuts=20, seed=seed)
        cv_s = CVPlus(alpha=ALPHA).fit(c_s.pred_oof, c_s.RUL, c_s.fold)
        mc_s = MondrianCV(alpha=ALPHA).fit(c_s.pred_oof, c_s.RUL)
        for k, d in EVAL.items():
            fp = fold_predictions(m_s, d[FEATS])
            for name, (lo, hi) in [("CV+", cv_s.predict(fp)), ("Mondrian CV", mc_s.predict(fp.mean(axis=1)))]:
                rep.append({"graine": seed, "méthode": name, "ensemble": k, "couverture": coverage(d.RUL, lo, hi), "largeur": mean_width(lo, hi)})
        print(f"répartition {seed} terminée")
    rep = pd.DataFrame(rep)
    print(rep.groupby(["méthode", "ensemble"])[["couverture", "largeur"]].agg(["mean", "min", "max"]).round(3))

## 10. Les intervalles au fil de la vie des moteurs

In [ ]:
full_val = sets["val"][sets["val"].cycle >= 31].copy()
fp_full = fold_predictions(models, full_val[FEATS])
full_val["pred"] = fp_full.mean(axis=1)
full_val["cv_lo"], full_val["cv_hi"] = cvp.predict(fp_full)
full_val["m_lo"], full_val["m_hi"] = mcv.predict(full_val.pred)
full_val["L"] = mcv.lower_bound(full_val.pred)

units = full_val.unit.unique()[:4]
fig, axes = plt.subplots(2, 4, figsize=(18, 7), sharey=True)
for j, u in enumerate(units):
    d = full_val[full_val.unit == u]
    for i, (m, lo, hi, c) in enumerate([("CV+", "cv_lo", "cv_hi", "steelblue"), ("Mondrian CV", "m_lo", "m_hi", "seagreen")]):
        ax = axes[i, j]
        ax.fill_between(d.cycle, d[lo], d[hi], color=c, alpha=0.25, label="intervalle 90 %")
        ax.plot(d.cycle, d.pred, color=c, lw=1, label="prédiction")
        ax.plot(d.cycle, d.RUL, "k--", lw=1, label="vraie RUL")
        if i == 1:
            ax.plot(d.cycle, d.L, color="darkred", lw=1, label="borne basse 95 %")
        ax.set(title=f"{m} – moteur {u}", xlabel="Cycle")
axes[0, 0].legend(fontsize=8)
axes[1, 0].legend(fontsize=8)
plt.tight_layout()
plt.show()

## 11. Sauvegarde (pour le tableau de bord de l'étape 6)

In [ ]:
joblib.dump({"models": models, "folds": folds, "features": FEATS, "cvplus": cvp, "mondrian": mcv,
             "calibration": cal[["unit", "cycle", "RUL", "pred_oof", "fold"]], "alpha": ALPHA},
            MODELS / "conformal_cv_fd001.joblib")
summary.to_csv(PROJECT_ROOT / "reports" / "conformal_summary_cv.csv", index=False)
print("Sauvegardé")

## 12. Conclusions

_À compléter :_
- De combien q variait-il avec 15 moteurs ? Et avec 85 ?
- La couverture est-elle tenue sur la validation ET le test, avec les deux méthodes ?
- Les intervalles sont-ils plus étroits qu'à l'étape 5 ?
- Les résultats changent-ils avec une autre répartition des moteurs (section 9) ?
- De combien la borne basse adaptative recule-t-elle en moyenne, contre 26,9 pour la globale ?